# Phân tích SQL - Dữ liệu thương mại điện tử Olist

Notebook này chạy 10 câu truy vấn Q01–Q10 từ tệp `sql/03_queries.sql` bằng SQLite và pandas.

### Mục tiêu phân tích
- Kiểm tra ảnh hưởng của ngày đặc biệt và biến động doanh số.
- Đánh giá phân bố và mức cân bằng của `review_score`.
- So sánh giao trễ, phí vận chuyển, độ phức tạp đơn hàng, thanh toán và khu vực với mức hài lòng.


## Tóm tắt và tổng hợp notebook

### Mục đích
Notebook phân tích dữ liệu Brazilian E-Commerce Public Dataset by Olist giai đoạn 2016–2018 nhằm trả lời hai câu hỏi trọng tâm:
- Những yếu tố nào trong quá trình mua và giao hàng chi phối mức độ hài lòng `review_score`?
- Có thể dự đoán review tại thời điểm đặt hàng hay chỉ sau khi giao hàng mà không vi phạm data leakage?

### Mục tiêu
Notebook kết nối SQLite với `data/olist.db`, tải view và đặc trưng từ `sql/03_queries.sql`, đồng thời tổng hợp item, thanh toán và người bán theo từng đơn hàng trước khi phân tích để tránh nhân đôi dữ liệu.

10 truy vấn được tổ chức thành các nhóm mục tiêu:
- **Q01–Q02:** Ngày đặc biệt, dịp mua sắm và phát hiện ngày có lượng đơn đột biến.
- **Q03:** Phân bố điểm đánh giá và mức mất cân bằng lớp.
- **Q04–Q05:** Mối quan hệ giữa giao trễ, mức độ trễ và điểm review.
- **Q06:** Tổng phí ship và tỷ lệ phí ship trên giá trị hàng hóa.
- **Q07:** Độ phức tạp qua số sản phẩm, số mặt hàng và số người bán trong đơn.
- **Q08:** Phương thức thanh toán và số kỳ trả góp.
- **Q09:** Xu hướng review theo thời gian 22 tháng.
- **Q10:** Phân hóa mức hài lòng theo bang và khu vực khách hàng.

### Đầu ra và cách sử dụng
- Mỗi truy vấn trả về một `DataFrame`, dùng cho so sánh, trực quan hóa và rút kết luận kinh doanh/logistics.
- `v_model_features_at_purchase` chỉ chứa thông tin có tại thời điểm đặt hàng và phù hợp cho mô hình dự đoán sớm.
- `v_model_features_post_delivery` bổ sung biến giao hàng và chỉ phù hợp khi dự đoán sau khi đơn hàng đã được giao.
- Việc tách hai tập feature giúp kiểm soát data leakage và hỗ trợ câu hỏi nghiên cứu thứ hai một cách minh bạch.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

base_dir = Path.cwd().resolve()
if base_dir.name == "Notebooks":
    base_dir = base_dir.parent

db_path = base_dir / "data" / "olist.db"
sql_path = base_dir / "sql" / "03_queries.sql"

if not db_path.exists():
    raise FileNotFoundError(f"Khong tim thay database: {db_path}")

print("Database:", db_path)
print("SQL source:", sql_path)

Database: E:\ADY291c-AI2107-Group2\data\olist.db
SQL source: E:\ADY291c-AI2107-Group2\sql\03_queries.sql


In [2]:
# Tạo các view cần thiết; không chạy các câu truy vấn Q01-Q10 trong file SQL.
# Mỗi câu truy vấn phân tích được thực thi riêng ở cell bên dưới.
sql_text = sql_path.read_text(encoding="utf-8")
q01_marker = "-- Q01:"
leakage_marker = "-- LEAKAGE-SAFE MODEL FEATURE VIEWS"

view_setup = sql_text.split(q01_marker, 1)[0]
if leakage_marker in sql_text:
    view_setup += "\n" + leakage_marker + "\n"
    view_setup += sql_text.split(leakage_marker, 1)[1]

conn = sqlite3.connect(db_path, timeout=30)
conn.execute("PRAGMA cache_size = -64000")
conn.execute("PRAGMA temp_store = MEMORY")
conn.executescript(view_setup)

views = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name"
).fetchall()

print("Danh sách view:")
for view_name, in views:
    print("-", view_name)

print("Số đơn phân tích:", conn.execute(
    "SELECT COUNT(*) FROM v_review_order_features"
).fetchone()[0])


Danh sách view:
- v_category_performance
- v_customer_rfm
- v_delivery_kpi
- v_model_features_at_purchase
- v_model_features_post_delivery
- v_payment_mix
- v_review_order_features
- v_sales_monthly
- v_seller_performance
- view_order_clean
Số đơn phân tích: 95329


## Quy ước dữ liệu

View `v_review_order_features` chứa một dòng cho mỗi đơn đã giao và đã có đánh giá. Dữ liệu sản phẩm và thanh toán được tổng hợp theo `order_id` trước khi JOIN để tránh nhân đôi đánh giá.

- `delay_days <= 0`: đơn giao đúng hạn hoặc sớm hơn dự kiến.
- `delay_days > 0`: đơn giao trễ so với ngày dự kiến.
- `review_score`: điểm đánh giá từ 1 đến 5.


## Q01: Ngay le va dip dac biet

**Mục tiêu:** So sánh số đơn mỗi ngày đặc biệt với trung bình số đơn của các ngày thường trong cùng năm.

**Cách thực hiện:** Cell SQL tính số đơn theo ngày, lấy danh sách ngày đặc biệt và tính phần trăm chênh lệch.

**Kết quả mong đợi:** DataFrame `df_q01` chứa dữ liệu đã tổng hợp.

In [3]:
q01 = """
WITH special_dates(special_date, special_name) AS (
    VALUES
        ('2016-02-08','Carnival'), ('2016-02-09','Carnival'),
        ('2016-11-25','Black Friday'), ('2016-12-25','Giang sinh'),
        ('2017-02-27','Carnival'), ('2017-02-28','Carnival'),
        ('2017-11-24','Black Friday'), ('2017-12-25','Giang sinh'),
        ('2018-02-12','Carnival'), ('2018-02-13','Carnival'),
        ('2018-03-30','Good Friday'), ('2018-05-31','Corpus Christi'),
        ('2018-11-23','Black Friday'), ('2018-12-25','Giang sinh')
),
daily_orders AS (
    SELECT date(order_purchase_timestamp) AS purchase_date,
           COUNT(DISTINCT order_id) AS order_count
    FROM orders
    GROUP BY date(order_purchase_timestamp)
),
normal_by_year AS (
    SELECT substr(purchase_date, 1, 4) AS purchase_year,
           AVG(order_count) AS avg_order_count
    FROM daily_orders
    WHERE purchase_date NOT IN (SELECT special_date FROM special_dates)
    GROUP BY substr(purchase_date, 1, 4)
)
SELECT d.purchase_date,
       s.special_name,
       d.order_count,
       ROUND(n.avg_order_count, 2) AS avg_normal_orders,
       ROUND(100.0 * (d.order_count - n.avg_order_count)
             / NULLIF(n.avg_order_count, 0), 2) AS pct_vs_normal_days
FROM daily_orders d
JOIN special_dates s ON s.special_date = d.purchase_date
JOIN normal_by_year n
  ON n.purchase_year = substr(d.purchase_date, 1, 4)
ORDER BY d.order_count DESC, d.purchase_date;
"""

df_q01 = pd.read_sql_query(q01, conn)
df_q01


,purchase_date,special_name,order_count,avg_normal_orders,pct_vs_normal_days
0,2017-11-24,Black Friday,1147,117.92,872.66
1,2018-02-13,Carnival,217,219.62,-1.19
2,2018-02-12,Carnival,214,219.62,-2.56
3,2018-03-30,Good Friday,163,219.62,-25.78
4,2018-05-31,Corpus Christi,132,219.62,-39.90
5,2017-12-25,Giang sinh,86,117.92,-27.07
6,2017-02-28,Carnival,53,117.92,-55.06
7,2017-02-27,Carnival,41,117.92,-65.23


## Q02: Phat hien ngay dot bien

**Mục tiêu:** Tìm những ngày có lượng đơn đột biến so với lịch sử ngắn hạn trước đó.

**Cách thực hiện:** Cửa sổ 28 ngày trước được dùng làm đường chuẩn; ngày được đánh dấu khi số đơn đạt ít nhất 150% mức trung bình trước đó.

**Kết quả mong đợi:** DataFrame `df_q02` chứa dữ liệu đã tổng hợp.

In [4]:
q02 = """
WITH daily_orders AS (
    SELECT date(order_purchase_timestamp) AS purchase_date,
           COUNT(DISTINCT order_id) AS order_count
    FROM orders
    GROUP BY date(order_purchase_timestamp)
),
baseline AS (
    SELECT purchase_date,
           order_count,
           AVG(order_count) OVER (
               ORDER BY purchase_date
               ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING
           ) AS prior_28d_avg
    FROM daily_orders
)
SELECT purchase_date,
       order_count,
       ROUND(prior_28d_avg, 2) AS prior_28d_avg,
       ROUND(100.0 * (order_count - prior_28d_avg)
             / NULLIF(prior_28d_avg, 0), 2) AS spike_pct
FROM baseline
WHERE prior_28d_avg IS NOT NULL
  AND order_count >= 1.50 * prior_28d_avg
ORDER BY spike_pct DESC, purchase_date;
"""

df_q02 = pd.read_sql_query(q02, conn)
df_q02


,purchase_date,order_count,prior_28d_avg,spike_pct
0,2016-10-04,54,7.00,671.43
1,2017-11-24,1147,163.21,602.76
2,2017-01-26,82,22.93,257.63
3,2017-02-07,108,39.25,175.16
4,2017-01-25,57,22.82,149.77
5,2017-01-31,64,25.75,148.54
6,2017-11-25,487,199.54,144.07
7,2017-01-27,59,24.61,139.77
8,2017-02-02,67,28.96,131.32
9,2017-02-01,61,26.82,127.43


## Q03: Phan bo va mat can bang review_score

**Mục tiêu:** Xem số lượng và tỷ lệ đơn theo từng mức đánh giá từ 1 đến 5.

**Cách thực hiện:** Kết quả cho biết nhóm điểm chiếm ưu thế và mức mất cân bằng trong dữ liệu đánh giá.

**Kết quả mong đợi:** DataFrame `df_q03` chứa dữ liệu đã tổng hợp.

In [5]:
q03 = """
SELECT review_score,
       COUNT(*) AS order_count,
       ROUND(100.0 * COUNT(*) /
             (SELECT COUNT(*) FROM v_review_order_features), 2) AS percentage
FROM v_review_order_features
GROUP BY review_score
ORDER BY review_score;
"""

df_q03 = pd.read_sql_query(q03, conn)
df_q03


,review_score,order_count,percentage
0,1,9277,9.73
1,2,2900,3.04
2,3,7865,8.25
3,4,18807,19.73
4,5,56480,59.25


## Q04: Don giao tre so voi dung han

**Mục tiêu:** So sánh tỷ lệ đánh giá tốt và không tốt giữa đơn giao trễ và đơn giao đúng hạn.

**Cách thực hiện:** Nhóm giao trễ được xác định bằng `delay_days > 0`.

**Kết quả mong đợi:** DataFrame `df_q04` chứa dữ liệu đã tổng hợp.

In [6]:
q04 = """
SELECT CASE WHEN is_late = 1 THEN 'Giao tre'
            ELSE 'Dung han hoac som' END AS delivery_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY is_late
ORDER BY delivery_group;
"""

df_q04 = pd.read_sql_query(q04, conn)
df_q04


,delivery_group,order_count,avg_review_score,positive_pct,low_score_pct
0,Dung han hoac som,88985,4.29,82.71,9.23
1,Giao tre,6344,2.27,26.66,62.42


## Q05: Muc do tre theo ngay

**Mục tiêu:** Phân tích mức độ giao trễ theo từng mốc thời gian đặt hàng.

**Cách thực hiện:** Câu truy vấn gom đơn theo ngày hoặc chu kỳ tương ứng để quan sát xu hướng trễ.

**Kết quả mong đợi:** DataFrame `df_q05` chứa dữ liệu đã tổng hợp.

In [7]:
q05 = """
SELECT CASE
           WHEN delay_days <= 0 THEN 'Som hoac dung han'
           WHEN delay_days <= 3 THEN 'Tre 1-3 ngay'
           WHEN delay_days <= 7 THEN 'Tre 4-7 ngay'
           ELSE 'Tre tren 7 ngay'
       END AS delay_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct
FROM v_review_order_features
WHERE delay_days IS NOT NULL
GROUP BY delay_group
ORDER BY MIN(CASE
                 WHEN delay_days <= 0 THEN 0
                 WHEN delay_days <= 3 THEN 1
                 WHEN delay_days <= 7 THEN 2
                 ELSE 3 END);
"""

df_q05 = pd.read_sql_query(q05, conn)
df_q05


,delay_group,order_count,avg_review_score,positive_pct
0,Som hoac dung han,88985,4.29,82.71
1,Tre 1-3 ngay,1841,3.29,53.18
2,Tre 4-7 ngay,1740,2.10,22.41
3,Tre tren 7 ngay,2763,1.69,11.65


## Q06: Phi ship va ty le phi/gia

**Mục tiêu:** Đánh giá mối liên hệ giữa phí vận chuyển, tỷ lệ phí/gía trị hàng và mức đánh giá.

**Cách thực hiện:** Dữ liệu được tổng hợp theo nhóm phân tích để kiểm tra chi phí và mức hài lòng.

**Kết quả mong đợi:** DataFrame `df_q06` chứa dữ liệu đã tổng hợp.

In [8]:
q06 = """
WITH freight_quartiles AS (
    SELECT *, NTILE(4) OVER (ORDER BY total_freight) AS group_number
    FROM v_review_order_features
),
ratio_quartiles AS (
    SELECT *, NTILE(4) OVER (ORDER BY freight_to_price_pct) AS group_number
    FROM v_review_order_features
    WHERE freight_to_price_pct IS NOT NULL
)
SELECT 'Tong phi ship' AS metric,
       group_number,
       COUNT(*) AS order_count,
       ROUND(AVG(total_freight), 2) AS avg_value,
       ROUND(AVG(review_score), 2) AS avg_review_score
FROM freight_quartiles
GROUP BY group_number
UNION ALL
SELECT 'Ty le phi/gia hang' AS metric,
       group_number,
       COUNT(*) AS order_count,
       ROUND(AVG(freight_to_price_pct), 2) AS avg_value,
       ROUND(AVG(review_score), 2) AS avg_review_score
FROM ratio_quartiles
GROUP BY group_number
ORDER BY metric, group_number;
"""

df_q06 = pd.read_sql_query(q06, conn)
df_q06


,metric,group_number,order_count,avg_value,avg_review_score
0,Tong phi ship,1,23833,10.44,4.32
1,Tong phi ship,2,23832,15.46,4.19
2,Tong phi ship,3,23832,19.77,4.18
3,Tong phi ship,4,23832,45.39,3.94
4,Ty le phi/gia hang,1,23833,8.41,4.20
5,Ty le phi/gia hang,2,23832,17.55,4.17
6,Ty le phi/gia hang,3,23832,29.18,4.15
7,Ty le phi/gia hang,4,23832,68.17,4.12


## Q07: Do phuc tap don hang

**Mục tiêu:** So sánh mức hài lòng theo số sản phẩm, số loại sản phẩm và số người bán trong đơn.

**Cách thực hiện:** Các đặc trưng của đơn được tổng hợp trước khi phân nhóm để tránh nhân đôi dữ liệu.

**Kết quả mong đợi:** DataFrame `df_q07` chứa dữ liệu đã tổng hợp.

In [9]:
q07 = """
SELECT CASE
           WHEN item_count = 1 THEN '1 mat hang'
           WHEN item_count BETWEEN 2 AND 3 THEN '2-3 mat hang'
           ELSE '4+ mat hang'
       END AS basket_group,
       CASE WHEN seller_count = 1 THEN '1 nguoi ban'
            ELSE 'Nhieu nguoi ban' END AS seller_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY basket_group, seller_group
ORDER BY MIN(item_count), MIN(seller_count);
"""

df_q07 = pd.read_sql_query(q07, conn)
df_q07


,basket_group,seller_group,order_count,avg_review_score,low_score_pct
0,1 mat hang,1 nguoi ban,85871,4.21,11.24
1,2-3 mat hang,1 nguoi ban,7408,3.78,22.80
2,2-3 mat hang,Nhieu nguoi ban,1132,2.89,46.11
3,4+ mat hang,1 nguoi ban,804,3.48,30.60
4,4+ mat hang,Nhieu nguoi ban,114,2.49,59.65


## Q08: Thanh toan va tra gop

**Mục tiêu:** Phân tích ảnh hưởng của hình thức thanh toán và số kỳ trả góp đến điểm đánh giá.

**Cách thực hiện:** Kết quả đối chiếu số đơn và điểm đánh giá trung bình trong từng nhóm thanh toán.

**Kết quả mong đợi:** DataFrame `df_q08` chứa dữ liệu đã tổng hợp.

In [10]:
q08 = """
SELECT first_payment_type AS payment_type,
       COUNT(*) AS order_count,
       ROUND(AVG(max_installments), 2) AS avg_installments,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
WHERE first_payment_type IS NOT NULL
GROUP BY first_payment_type
ORDER BY order_count DESC;
"""

df_q08 = pd.read_sql_query(q08, conn)
df_q08


,payment_type,order_count,avg_installments,avg_review_score,low_score_pct
0,credit_card,73387,3.5,4.16,12.84
1,boleto,18970,1.0,4.16,12.51
2,voucher,1471,1.0,4.11,14.28
3,debit_card,1424,1.0,4.25,10.88


## Q09: Xu huong review theo thang

**Mục tiêu:** Quan sát biến động điểm đánh giá trung bình và số lượng review theo từng tháng.

**Cách thực hiện:** Kết quả được sắp xếp theo năm-tháng để nhận diện xu hướng thời gian.

**Kết quả mong đợi:** DataFrame `df_q09` chứa dữ liệu đã tổng hợp.

In [11]:
q09 = """
SELECT purchase_year_month,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY purchase_year_month
ORDER BY purchase_year_month;
"""

df_q09 = pd.read_sql_query(q09, conn)
df_q09


,purchase_year_month,order_count,avg_review_score,positive_pct,low_score_pct
0,2016-10,260,4.01,75.38,16.54
1,2016-12,1,5.00,100.00,0.00
2,2017-01,729,4.20,79.84,11.93
3,2017-02,1634,4.21,79.44,11.02
4,2017-03,2506,4.19,79.93,11.13
5,2017-04,2276,4.13,78.08,13.01
6,2017-05,3490,4.24,80.97,10.00
7,2017-06,3083,4.22,80.60,10.70
8,2017-07,3809,4.26,81.46,9.90
9,2017-08,4127,4.32,83.81,9.33


## Q10: Hai long theo bang/khu vuc

**Mục tiêu:** So sánh mức hài lòng theo bang hoặc khu vực của khách hàng.

**Cách thực hiện:** Kết quả gom dữ liệu theo địa lý để tìm khu vực có đánh giá cao hoặc thấp.

**Kết quả mong đợi:** DataFrame `df_q10` chứa dữ liệu đã tổng hợp.

In [12]:
q10 = """
SELECT customer_state,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct,
       ROUND(AVG(is_late) * 100.0, 2) AS late_rate_pct,
       ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM v_review_order_features
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY avg_review_score ASC, low_score_pct DESC;
"""

df_q10 = pd.read_sql_query(q10, conn)
df_q10


,customer_state,order_count,avg_review_score,positive_pct,low_score_pct,late_rate_pct,avg_delivery_days
0,MA,706,3.82,69.97,20.11,17.28,21.44
1,AL,393,3.85,71.76,21.12,20.61,24.41
2,SE,332,3.90,72.29,18.98,15.06,21.46
3,PA,924,3.92,72.40,17.86,10.82,23.64
4,BA,3213,3.93,72.92,16.99,11.86,19.23
5,CE,1272,3.94,72.80,17.14,13.68,21.22
6,RJ,12136,3.97,73.48,18.27,11.93,15.24
7,PI,470,3.99,75.11,16.17,13.62,19.47
8,PB,510,4.07,75.88,14.90,10.20,20.26
9,PE,1573,4.08,77.24,14.88,9.47,18.38


In [13]:
# Đóng kết nối sau khi hoàn tất toàn bộ truy vấn.
conn.close()
